# Hands-On with Diffusion Models — Part 2
**IndabaX Uganda 2026 | Stable Diffusion & LoRA Style Fine-Tuning**

In this session we build directly on Part 1:

- We implemented DDPM **from scratch** — noise schedule, training loop, sampling, context mixing.
- Now we use **Hugging Face Diffusers** to work with Stable Diffusion XL (SDXL), a production model.
- We fine-tune it for **artistic style** using **LoRA** — the same concept as your context vector, scaled up enormously.
- The payoff: **style mixing** — the same weight slider you used for `female_weight`, but now blending art movements.

**Session arc** (mirrors Part 1, one level up):

| Part 1 | Part 2 |
|--------|--------|
| Understand the math | Understand what Diffusers gives you |
| Implement forward process | Run inference on a pretrained model |
| Implement training loop | Understand LoRA conceptually |
| Sample and visualise | Fine-tune and observe |
| Context mixing payoff | **Style mixing payoff** |

**Requirements:** GPU ≥16 GB VRAM · Python 3.10+ · Diffusers / PEFT / Datasets

In [ ]:
# Standard library
import os
import math
import random
from pathlib import Path

# Deep learning
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms

# Diffusers / HuggingFace
from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel
from peft import LoraConfig

# Visualisation
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# ── Part 2 modules (dataset.py, model.py, utils.py) ──────────────────────────
from dataset import WikiArtStyleDataset, load_wikiart
from model   import load_sdxl_components, configure_lora
from utils   import *
# ── Device setup ─────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16   # RTX 5090 (Blackwell) natively supports bfloat16

print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
config = load_config("config.yaml")
print("Config loaded ✓")

---
## Part 0: Why Diffusers?

You spent Part 1 writing ~220 lines of carefully crafted Python to train and sample from a DDPM.
Now watch what Diffusers lets you do.

In [ ]:
print("Loading SDXL pipeline …")
pipe_base = StableDiffusionXLPipeline.from_pretrained(
    config["model_id"], torch_dtype=DTYPE, use_safetensors=True, variant="fp16",
).to(device)
pipe_base.set_progress_bar_config(disable=True)

generator = torch.Generator(device=device).manual_seed(config["seed"])
with torch.no_grad():
    image_base = pipe_base(
        prompt              = config["comparison_prompt"],
        negative_prompt     = config["negative_prompt"],
        num_inference_steps = config["num_inference_steps"],
        guidance_scale      = config["guidance_scale"],
        height=config["image_size"], width=config["image_size"],
        generator=generator,
    ).images[0]

plt.figure(figsize=(7, 7))
plt.imshow(image_base)
plt.axis("off")
plt.title("SDXL base — no fine-tuning", fontsize=11)
plt.tight_layout()
plt.show()

---
## Part 1: WikiArt — The Training Dataset

### Why WikiArt?

In Part 1 we conditioned on **gender** (a binary label: female / male).
Now we scale up: **artistic style** across 27 movements — from Impressionism and Baroque to Ukiyo-e and Surrealism.

The WikiArt dataset on Hugging Face Hub (`huggan/wikiart`) is the ideal choice:
- **~80 000 paintings** from 129 artists, spanning five centuries of art history.
- Three rich label columns: `artist`, `style` (27 classes), `genre` (11 classes).
- Loads in one line of code — no manual download, no folder restructuring.

The style column is the direct pedagogical continuation of the gender context from Part 1:
it is a richer, more nuanced conditioning signal, one the model must *learn* rather than simply embed.

In [ ]:
train_ds = load_wikiart(config["dataset_path"])

In [ ]:
# ── Extract style metadata ────────────────────────────────────────────────────
style_feature = train_ds.features["style"]
style_names   = style_feature.names          # list of 27 style name strings
style_ids     = train_ds["style"]            # list of integer labels

# Count images per style
from collections import Counter
style_counts = Counter(style_ids)
sorted_items = sorted(style_counts.items(), key=lambda x: -x[1])

labels  = [style_names[i].replace("_", " ") for i, _ in sorted_items]
counts  = [c for _, c in sorted_items]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(labels, counts, color=plt.cm.tab20(np.linspace(0, 1, len(labels))))
ax.set_xlabel("Number of paintings")
ax.set_title("WikiArt: image count by artistic style (27 classes)", fontsize=13)
ax.invert_yaxis()

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f"{count:,}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print("Full style list:")
for i, name in enumerate(style_names):
    print(f"  {i:2d}: {name.replace('_', ' ')}")

In [ ]:
# ── Visualise sample images from several styles ───────────────────────────────
PREVIEW_STYLES = [style_names[id] for id, count in sorted_items[:5]]
N_PER_STYLE    = 4

fig, axes = plt.subplots(len(PREVIEW_STYLES), N_PER_STYLE,
                          figsize=(N_PER_STYLE * 3, len(PREVIEW_STYLES) * 3))

for row, style_name in enumerate(PREVIEW_STYLES):
    style_id = style_names.index(style_name)
    # Collect indices for this style
    indices  = [i for i, s in enumerate(style_ids) if s == style_id]
    chosen   = random.sample(indices, min(N_PER_STYLE, len(indices)))

    for col, idx in enumerate(chosen):
        img = train_ds[idx]["image"].convert("RGB")
        img.thumbnail((256, 256))
        axes[row, col].imshow(img)
        axes[row, col].set_title(style_name)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(style_name.replace("_", " "), fontsize=10, rotation=0,
                                       labelpad=90, va="center")

plt.suptitle("WikiArt sample images by style", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
STYLE_A_ID = style_names.index(config["style_a_name"])
STYLE_B_ID = style_names.index(config["style_b_name"])

print(f"Style A — {config['style_a_name']}: {style_counts[STYLE_A_ID]:,} paintings (index {STYLE_A_ID})")
print(f"Style B — {config['style_b_name']}: {style_counts[STYLE_B_ID]:,} paintings (index {STYLE_B_ID})")
print("\nWe train a separate LoRA for each style, then blend them in Part 4.")
print("The blending weight is the direct analogue of `female_weight` from Part 1.")

---
## Part 2: Understanding LoRA

### What is LoRA and why does it matter?

Stable Diffusion XL has **860 million parameters**.
A full fine-tune would require:
- Storing a complete copy of all gradients (~3 × model size in memory).
- Saving a ~6 GB checkpoint per style.
- Hours of training even on high-end hardware.

**Low-Rank Adaptation (LoRA)** sidesteps this entirely.
Instead of updating every weight matrix $W$, we inject two tiny matrices $A$ and $B$ and only train those:

$$W' = W + \Delta W = W + BA$$

where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, and $r \ll \min(d, k)$.

The rank $r$ is a hyperparameter (typically 4–64).
The base weights $W$ are **frozen** — they are never updated.

### The numbers speak for themselves

| | Full fine-tune | LoRA (rank 16) |
|---|---|---|
| Trainable params | 860 M | ~3 M |
| Checkpoint size | ~6.5 GB | ~25 MB |
| Training time | Hours | Minutes |
| Multiple styles | Separate 6 GB file per style | Stack adapters, blend with weights |

### Where does LoRA live in the architecture?

LoRA is injected into the **attention layers** of the U-Net:
specifically the query, key, value, and output projection matrices (`to_q`, `to_k`, `to_v`, `to_out`).

**Connection to Part 1:** In your custom `DDPMUnet`, the context conditioning is injected at the upsampling path:

```python
up2 = self.up1(cemb1 * up1 + temb1, down2)   # context embedding modulates features
up3 = self.up2(cemb2 * up2 + temb2, down1)
```

This is **cross-attention** — the context vector steers the feature maps.
In SDXL, cross-attention between text embeddings and image features is exactly what LoRA targets.
You already understand the concept.  LoRA just makes modifying those weights efficient.

### The analogy

> The base SDXL model is a vast artistic vocabulary.
> The LoRA is a **thin specialised lens** you place in front of it.
> The lens does not replace the vocabulary — it redirects it toward a style.

In [ ]:
unet_for_count = UNet2DConditionModel.from_pretrained(
    config["model_id"], subfolder="unet", torch_dtype=DTYPE
)
total_params = sum(p.numel() for p in unet_for_count.parameters())

unet_for_count.add_adapter(LoraConfig(
    r=config["lora_rank"], lora_alpha=config["lora_alpha"],
    init_lora_weights="gaussian", target_modules=config["lora_target_modules"],
))
lora_params = sum(p.numel() for n, p in unet_for_count.named_parameters() if "lora" in n.lower())
print(f"UNet total : {total_params:>12,}")
print(f"LoRA only  : {lora_params:>12,}  ({100*lora_params/total_params:.2f}% of the model)")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Full fine-tune", f"LoRA (rank {config['lora_rank']})"],
              [total_params/1e6, lora_params/1e6], color=["#e74c3c", "#2ecc71"], width=0.4)
for bar, val in zip(bars, [total_params/1e6, lora_params/1e6]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f"{val:.1f}M", ha="center", fontweight="bold")
ax.set_ylabel("Parameters (millions)"); ax.set_yscale("log")
ax.set_title("Trainable parameters: full fine-tune vs LoRA")
plt.tight_layout(); plt.show()

del unet_for_count
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## Part 3a: Impressionism LoRA Fine-Tuning

### The plan

We train the **first LoRA** on Impressionism paintings from WikiArt.

**Training recipe:**
1. Filter WikiArt to Impressionism images.
2. Generate captions: `"a painting in the style of impressionism"`.
3. Encode images → latent space (VAE).
4. Encode captions → text embeddings (two CLIP encoders).
5. Add noise at a random timestep.
6. Forward pass through the frozen UNet + active LoRA.
7. MSE loss between predicted noise and actual noise.
8. Backprop through LoRA matrices only.
9. Save adapter weights — periodic checkpoints **and** a best-loss checkpoint.

After training we immediately test the Impressionism LoRA before moving on to Ukiyo-e.

> **Live demo tip:** Start training running now, walk through the theory above, then come back to see the first checkpoint.

In [ ]:
ds_impressionism = WikiArtStyleDataset(train_ds, STYLE_A_ID, config["style_a_name"], image_size=config["image_size"])
ds_ukiyo_e       = WikiArtStyleDataset(train_ds, STYLE_B_ID, config["style_b_name"], image_size=config["image_size"])

sample = ds_impressionism[0]
print(f"Image shape : {sample['image'].shape}")
print(f"Caption     : {sample['caption']}")

In [ ]:
noise_scheduler, tokenizer_1, tokenizer_2, text_encoder_1, text_encoder_2, vae, unet = \
    load_sdxl_components(config["model_id"], DTYPE, device)

In [ ]:
unet, lora_config = configure_lora(
    unet,
    rank           = config["lora_rank"],
    alpha          = config["lora_alpha"],
    target_modules = config["lora_target_modules"],
)

### Training helpers (from `utils.py`)

The functions used inside the training loop are defined in `utils.py`:

| Function | Purpose |
|----------|---------|
| `encode_prompt(captions, ...)` | Tokenise and embed text with both SDXL CLIP encoders |
| `get_add_time_ids(batch_size, ...)` | Build SDXL's image-size conditioning tensor |
| `train_epoch(unet, dataloader, ...)` | One epoch: VAE encode → add noise → UNet forward → MSE loss → backprop |

Open `utils.py` to see the full implementations with step-by-step comments.

In [ ]:
SKIP_TRAINING_A = True   # ← Set to False to train

if not SKIP_TRAINING_A:
    loader_a = DataLoader(
        WikiArtStyleDataset(train_ds, STYLE_A_ID, config["style_a_name"],
                            image_size=config["image_size"], max_samples=config["max_samples"]),
        batch_size=config["batch_size"], shuffle=True,
        num_workers=config["num_workers"], pin_memory=True,
    )
    optimizer_a = torch.optim.AdamW(
        [p for p in unet.parameters() if p.requires_grad],
        lr=config["learning_rate"], weight_decay=1e-2,
    )
    save_dir_a    = Path(config["save_dir"]) / config["style_a_name"].lower()
    loss_history_a = []
    best_loss_a    = float("inf")

    for epoch in range(1, config["num_epochs"] + 1):
        avg_loss = train_epoch(
            unet, loader_a, optimizer_a, noise_scheduler, vae,
            tokenizer_1, tokenizer_2, text_encoder_1, text_encoder_2,
            device, DTYPE, epoch, config["style_a_name"], image_size=config["image_size"],
        )
        loss_history_a.append(avg_loss)
        print(f"  Epoch {epoch}/{config['num_epochs']}  loss = {avg_loss:.4f}")

        if epoch % config["save_every_n_epochs"] == 0 or epoch == config["num_epochs"]:
            save_lora_checkpoint(unet, save_dir_a / f"epoch_{epoch}")
            print(f"    → epoch_{epoch}/ saved")
        if avg_loss < best_loss_a:
            best_loss_a = avg_loss
            save_lora_checkpoint(unet, save_dir_a / "best")
            print(f"    ✓ New best: {best_loss_a:.4f}")

    out = save_lora_checkpoint(unet, save_dir_a / "final")
    print(f"\n✓ Done — best loss: {best_loss_a:.4f}  ({out.stat().st_size/1e6:.1f} MB)")
    plot_loss_curve(loss_history_a, config["style_a_name"], color="steelblue")
else:
    print(f"Skipping — weights at: {Path(config['save_dir']) / config['style_a_name'].lower() / 'best'}")

### Test: Impressionism LoRA

We load the best checkpoint and check two things:
1. **Style check** — does the LoRA produce impressionistic images?
2. **Before / after** — same prompt as Part 0, base model vs LoRA side by side.

In [ ]:
lora_path_a = Path(config["save_dir"]) / config["style_a_name"].lower() / "best"
assert lora_path_a.exists(), f"No weights at {lora_path_a} — run training first."

pipe_test_a = StableDiffusionXLPipeline.from_pretrained(
    config["model_id"], torch_dtype=DTYPE, use_safetensors=True, variant="fp16",
).to(device)
pipe_test_a.set_progress_bar_config(disable=True)
pipe_test_a.load_lora_weights(str(lora_path_a), adapter_name="style")
pipe_test_a.set_adapters(["style"], adapter_weights=[1.0])

# Style check — several images with the test prompt
images = []
for i in range(config["n_test_images"]):
    generator = torch.Generator(device=device).manual_seed(config["seed"] + i)
    with torch.no_grad():
        images.append(pipe_test_a(
            prompt=config["test_prompt"], negative_prompt=config["negative_prompt"],
            num_inference_steps=config["num_inference_steps"], guidance_scale=config["guidance_scale"],
            height=config["image_size"], width=config["image_size"], generator=generator,
        ).images[0])

ncols = min(config["n_test_images"], 4)
fig, axes = plt.subplots(1, ncols, figsize=(ncols*5, 5))
for ax, img in zip(axes, images):
    ax.imshow(img); ax.axis("off")
fig.suptitle(f"{config['style_a_name']} LoRA — \"{config['test_prompt']}\"", fontsize=11)
plt.tight_layout(); plt.show()

# Before / after — same prompt as Part 0, base model vs LoRA
generator = torch.Generator(device=device).manual_seed(config["seed"])
with torch.no_grad():
    image_lora_a = pipe_test_a(
        prompt=config["comparison_prompt"], negative_prompt=config["negative_prompt"],
        num_inference_steps=config["num_inference_steps"], guidance_scale=config["guidance_scale"],
        height=config["image_size"], width=config["image_size"], generator=generator,
    ).images[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(image_base); axes[0].axis("off"); axes[0].set_title("Base SDXL")
axes[1].imshow(image_lora_a); axes[1].axis("off"); axes[1].set_title(f"+ {config['style_a_name']} LoRA")
fig.suptitle(f'"{config["comparison_prompt"]}"', fontsize=11)
plt.tight_layout(); plt.show()

del pipe_test_a
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## Part 3b: Ukiyo-e LoRA Fine-Tuning

Now we train a **second, independent LoRA** for the Ukiyo-e style.

- The UNet's LoRA adapter is reset to fresh random weights — the Impressionism weights are already saved.
- The frozen base SDXL weights remain unchanged throughout.
- Ukiyo-e is chosen for **maximum visual contrast** with Impressionism:

| Impressionism | Ukiyo-e |
|---|---|
| Soft gradients | Flat areas of colour |
| Visible brush strokes | Bold outlines |
| Atmospheric perspective | Decorative composition |

This contrast will make the style-mixing payoff in Part 4 as striking as possible.

> After training, we test this LoRA individually before blending both styles in Part 4.

In [ ]:
SKIP_TRAINING_B = True   # ← Set to False to train

if not SKIP_TRAINING_B:
    # Reload UNet fresh — ensures no Impressionism weights carry over.
    del unet
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    unet = UNet2DConditionModel.from_pretrained(
        config["model_id"], subfolder="unet", torch_dtype=DTYPE
    ).to(device)
    unet, lora_config = configure_lora(
        unet, rank=config["lora_rank"], alpha=config["lora_alpha"],
        target_modules=config["lora_target_modules"],
    )

    loader_b = DataLoader(
        WikiArtStyleDataset(train_ds, STYLE_B_ID, config["style_b_name"],
                            image_size=config["image_size"], max_samples=config["max_samples"]),
        batch_size=config["batch_size"], shuffle=True,
        num_workers=config["num_workers"], pin_memory=True,
    )
    optimizer_b = torch.optim.AdamW(
        [p for p in unet.parameters() if p.requires_grad],
        lr=config["learning_rate"], weight_decay=1e-2,
    )
    save_dir_b    = Path(config["save_dir"]) / config["style_b_name"].lower()
    loss_history_b = []
    best_loss_b    = float("inf")

    for epoch in range(1, config["num_epochs"] + 1):
        avg_loss = train_epoch(
            unet, loader_b, optimizer_b, noise_scheduler, vae,
            tokenizer_1, tokenizer_2, text_encoder_1, text_encoder_2,
            device, DTYPE, epoch, config["style_b_name"], image_size=config["image_size"],
        )
        loss_history_b.append(avg_loss)
        print(f"  Epoch {epoch}/{config['num_epochs']}  loss = {avg_loss:.4f}")

        if epoch % config["save_every_n_epochs"] == 0 or epoch == config["num_epochs"]:
            save_lora_checkpoint(unet, save_dir_b / f"epoch_{epoch}")
            print(f"    → epoch_{epoch}/ saved")
        if avg_loss < best_loss_b:
            best_loss_b = avg_loss
            save_lora_checkpoint(unet, save_dir_b / "best")
            print(f"    ✓ New best: {best_loss_b:.4f}")

    out = save_lora_checkpoint(unet, save_dir_b / "final")
    print(f"\n✓ Done — best loss: {best_loss_b:.4f}  ({out.stat().st_size/1e6:.1f} MB)")
    plot_loss_curve(loss_history_b, config["style_b_name"], color="darkorange")
else:
    print(f"Skipping — weights at: {Path(config['save_dir']) / config['style_b_name'].lower() / 'best'}")

### Test: Ukiyo-e LoRA

Same checks as above — style verification and before/after comparison with the Part 0 prompt.

In [ ]:
lora_path_b = Path(config["save_dir"]) / config["style_b_name"].lower() / "best"
assert lora_path_b.exists(), f"No weights at {lora_path_b} — run training first."

pipe_test_b = StableDiffusionXLPipeline.from_pretrained(
    config["model_id"], torch_dtype=DTYPE, use_safetensors=True, variant="fp16",
).to(device)
pipe_test_b.set_progress_bar_config(disable=True)
pipe_test_b.load_lora_weights(str(lora_path_b), adapter_name="style")
pipe_test_b.set_adapters(["style"], adapter_weights=[1.0])

# Style check
images = []
for i in range(config["n_test_images"]):
    generator = torch.Generator(device=device).manual_seed(config["seed"] + i)
    with torch.no_grad():
        images.append(pipe_test_b(
            prompt=config["test_prompt"], negative_prompt=config["negative_prompt"],
            num_inference_steps=config["num_inference_steps"], guidance_scale=config["guidance_scale"],
            height=config["image_size"], width=config["image_size"], generator=generator,
        ).images[0])

ncols = min(config["n_test_images"], 4)
fig, axes = plt.subplots(1, ncols, figsize=(ncols*5, 5))
for ax, img in zip(axes, images):
    ax.imshow(img); ax.axis("off")
fig.suptitle(f"{config['style_b_name']} LoRA — \"{config['test_prompt']}\"", fontsize=11)
plt.tight_layout(); plt.show()

# Before / after — same prompt as Part 0
generator = torch.Generator(device=device).manual_seed(config["seed"])
with torch.no_grad():
    image_lora_b = pipe_test_b(
        prompt=config["comparison_prompt"], negative_prompt=config["negative_prompt"],
        num_inference_steps=config["num_inference_steps"], guidance_scale=config["guidance_scale"],
        height=config["image_size"], width=config["image_size"], generator=generator,
    ).images[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(image_base); axes[0].axis("off"); axes[0].set_title("Base SDXL")
axes[1].imshow(image_lora_b); axes[1].axis("off"); axes[1].set_title(f"+ {config['style_b_name']} LoRA")
fig.suptitle(f'"{config["comparison_prompt"]}"', fontsize=11)
plt.tight_layout(); plt.show()

del pipe_test_b
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## Part 4: Style Mixing — The Payoff

### The slider is back

In Part 1, you controlled gender with a single weight:

```python
ctx_mixed = torch.tensor([female_weight, 1 - female_weight])
```

At `female_weight = 1.0` → pure female.  At `0.0` → pure male.  At `0.5` → ambiguous blend.

Today, we do the exact same thing with art movements:

```python
pipe.set_adapters(["impressionism", "ukiyo_e"], adapter_weights=[w_imp, w_ukiyo])
```

At `[1.0, 0.0]` → pure Impressionism.
At `[0.0, 1.0]` → pure Ukiyo-e.
At `[0.5, 0.5]` → something genuinely surprising that neither style alone could produce.

### Why the slider, not just text?

We use **both**:
- The **text prompt** describes the **content** (subject matter, composition).
- The **LoRA weight slider** controls the **style** (learned visual language).

This separation is clean: the text does not know about Impressionism or Ukiyo-e —
only the LoRA adapter carries that knowledge.
The slider controls how much of each adapter's knowledge is activated.

In [ ]:
try:
    del unet, optimizer_a, optimizer_b, loader_a, loader_b
    del text_encoder_1, text_encoder_2, vae
except NameError:
    pass
torch.cuda.empty_cache() if torch.cuda.is_available() else None

pipe = StableDiffusionXLPipeline.from_pretrained(
    config["model_id"], torch_dtype=DTYPE, use_safetensors=True, variant="fp16",
).to(device)
pipe.set_progress_bar_config(disable=True)

lora_path_a = str(Path(config["save_dir"]) / config["style_a_name"].lower() / "best")
lora_path_b = str(Path(config["save_dir"]) / config["style_b_name"].lower() / "best")

pipe.load_lora_weights(lora_path_a, adapter_name="impressionism")
pipe.load_lora_weights(lora_path_b, adapter_name="ukiyo_e")
print(f"✓ Both LoRAs loaded — ready to blend")

In [ ]:
plot_style_sweep(
    pipe, device,
    prompt          = config["sweep_prompt"],
    negative_prompt = config["negative_prompt"],
    n_steps         = config["num_inference_steps"],
    guidance_scale  = config["guidance_scale"],
    seed            = config["seed"],
    image_size      = config["image_size"],
)

In [ ]:
# Pure Impressionism
generate_styled(pipe, device,
    prompt="a village street in summer, flowers, warm golden light",
    impressionism_weight=1.0,
    n_images=config["n_test_images"], n_steps=config["num_inference_steps"],
    guidance_scale=config["guidance_scale"], seed=config["seed"],
    negative_prompt=config["negative_prompt"], image_size=config["image_size"],
)

In [ ]:
# Pure Ukiyo-e
generate_styled(pipe, device,
    prompt="a village street in summer, flowers, warm golden light",
    impressionism_weight=0.0,
    n_images=config["n_test_images"], n_steps=config["num_inference_steps"],
    guidance_scale=config["guidance_scale"], seed=config["seed"],
    negative_prompt=config["negative_prompt"], image_size=config["image_size"],
)

In [ ]:
# 50 / 50 blend
generate_styled(pipe, device,
    prompt="a village street in summer, flowers, warm golden light",
    impressionism_weight=0.5,
    n_images=config["n_test_images"], n_steps=config["num_inference_steps"],
    guidance_scale=config["guidance_scale"], seed=config["seed"],
    negative_prompt=config["negative_prompt"], image_size=config["image_size"],
)

In [ ]:
# Try your own — change the prompt and weight below
generate_styled(pipe, device,
    prompt="a lone boat on a calm sea at dusk",
    impressionism_weight=0.3,    # ← try any value between 0 and 1
    n_images=config["n_test_images"], n_steps=40,
    guidance_scale=config["guidance_scale"], seed=config["seed"],
    negative_prompt=config["negative_prompt"], image_size=config["image_size"],
)

---
## Summary

### What we built today

| Step | Concept | Connection to Part 1 |
|------|---------|----------------------|
| Loaded SDXL | Diffusers abstracts what you implemented by hand | The 5-line version of your 220-line DDPM |
| Explored WikiArt | Richer conditioning signal than gender labels | Same idea, 27 classes instead of 2 |
| Understood LoRA | Efficient fine-tuning via low-rank matrices | Targets cross-attention — same layers your context embedding used |
| Trained two LoRAs | One per style, ~25 MB each | Analogous to training with class labels |
| Blended styles | `adapter_weights=[w1, w2]` | Exact analogue of `female_weight` slider |

### Key takeaways

1. **LoRA is the dominant fine-tuning paradigm** for diffusion models in production.
   File sizes are shareable. Multiple adapters can be stacked and blended at inference time.

2. **The cross-attention mechanism is where conditioning lives** — in your toy U-Net *and* in SDXL.
   LoRA targets exactly those layers because that is where style information is encoded.

3. **Full fine-tuning is rarely the right choice.**
   For style, concept, or subject adaptation, LoRA achieves comparable quality in a fraction of the compute.

4. **The slider intuition generalises.**
   Any two LoRAs trained on the same base model can be blended with weights.
   This extends to three or more adapters, enabling combinatorial style spaces.

---

## Acknowledgements

- **WikiArt dataset**: [huggan/wikiart](https://huggingface.co/datasets/huggan/wikiart)
- **Stable Diffusion XL**: Podell et al., *SDXL: Improving Latent Diffusion Models for High-Resolution Image Synthesis*, 2023
- **LoRA**: Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, 2021
- **Diffusers library**: Hugging Face — [github.com/huggingface/diffusers](https://github.com/huggingface/diffusers)
- **PEFT library**: Hugging Face — [github.com/huggingface/peft](https://github.com/huggingface/peft)

*Tutorial by IndabaX Uganda 2026.*